# 🤖 LLM Fine-tuning with QLoRA for Research Q&A

This notebook fine-tunes a large language model (Llama/Mistral) on YOUR research papers using **QLoRA** (Quantized Low-Rank Adaptation).

## Why Fine-tune the LLM?

**Generic LLMs** (like Gemini, GPT) are good at general Q&A but:
- May hallucinate facts about your research
- Don't understand your domain's nuances
- Can be expensive (API costs)

**YOUR fine-tuned LLM** will:
- Generate more accurate, domain-specific answers
- Better understand research paper structure
- Run locally (no API costs!)
- Be tailored to your research area

## What is QLoRA?

**QLoRA** = Quantized Low-Rank Adaptation

**Problem**: Fine-tuning a 7B parameter model normally requires 100GB+ GPU RAM

**QLoRA solution**:
- Quantizes model to 4-bit (massive memory reduction)
- Only trains small adapter layers (LoRA)
- Runs on free Colab GPU! (16GB)

**Result**: Fine-tune 7B models on consumer hardware!

## Requirements

- GPU with 16GB+ RAM (Colab free tier works!)
- Training data (Q&A pairs from your papers)
- 2-4 hours training time

---

**Let's build YOUR custom research assistant LLM!**

## Step 1: Setup and Install Dependencies

In [ ]:
# Install required packages
!pip install -q transformers datasets accelerate peft bitsandbytes trl

In [ ]:
import sys
import os
from pathlib import Path

# Add project to path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

import json
import torch
from datetime import datetime
from tqdm.auto import tqdm

# Transformers and training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer
from datasets import Dataset

from src.training.data_generator import TrainingDataGenerator

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("\n⚠️ WARNING: No GPU detected!")
    print("LLM fine-tuning REQUIRES a GPU. Please use Google Colab with GPU runtime.")
    print("\nTo enable GPU in Colab:")
    print("  Runtime → Change runtime type → Hardware accelerator → GPU")

## Step 2: Generate Q&A Training Data

We'll automatically generate question-answer pairs from your research papers.

In [ ]:
# Check if Q&A training data exists
qa_data_path = project_root / "data/training/qa_pairs.json"

if not qa_data_path.exists():
    print("Q&A training data not found. Generating...\n")
    
    # Load environment
    from dotenv import load_dotenv
    load_dotenv(project_root / ".env")
    
    if not os.getenv('GOOGLE_API_KEY'):
        raise ValueError(
            "GOOGLE_API_KEY required to generate Q&A pairs. "
            "Set it in .env file."
        )
    
    # Generate Q&A pairs
    generator = TrainingDataGenerator()
    
    qa_pairs = generator.generate_qa_pairs(
        papers_dir=str(project_root / "data/papers"),
        questions_per_chunk=3,
        output_file=str(qa_data_path)
    )
    
    print(f"\n✓ Generated {len(qa_pairs)} Q&A pairs")
else:
    print(f"✓ Q&A data found at {qa_data_path}")

# Load Q&A data
with open(qa_data_path) as f:
    qa_pairs = json.load(f)

print(f"\nLoaded {len(qa_pairs)} Q&A training pairs")
print(f"\nExample:")
print(f"Q: {qa_pairs[0]['question']}")
print(f"A: {qa_pairs[0]['answer'][:100]}...")

## Step 3: Prepare Training Dataset

Format Q&A pairs into instruction-following format for the LLM.

In [ ]:
# Format for instruction tuning
def format_instruction(example):
    """
    Format Q&A pair as instruction-following example.
    
    Template:
    [INST] <<SYS>>You are a research assistant...<</SYS>>
    Question: {question}
    Context: {context} [/INST]
    Answer: {answer}
    """
    system_prompt = (
        "You are a helpful research assistant specialized in analyzing research papers. "
        "Answer questions based on the provided context from research papers. "
        "Be accurate, concise, and cite specific details."
    )
    
    instruction = (
        f"[INST] <<SYS>>\n{system_prompt}\n<</SYS>>\n\n"
        f"Question: {example['question']}\n\n"
        f"Context: {example['context']}\n[/INST]\n\n"
        f"Answer: {example['answer']}"
    )
    
    return {'text': instruction}

# Convert to HuggingFace Dataset
dataset = Dataset.from_list(qa_pairs)
dataset = dataset.map(format_instruction)

# Split train/validation
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']

print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(eval_dataset)}")
print(f"\nExample formatted instruction:")
print(train_dataset[0]['text'][:300] + "...")

## Step 4: Load Base Model with 4-bit Quantization

We'll use **Llama 3.2** or **Mistral** as the base model.

In [ ]:
# Model selection (choose one)
MODEL_NAME = "meta-llama/Llama-3.2-3B"  # or "mistralai/Mistral-7B-v0.1"

print(f"Loading base model: {MODEL_NAME}")
print("This may take 5-10 minutes...\n")

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✓ Model loaded")
print(f"  Parameters: {model.num_parameters() / 1e9:.2f}B")
print(f"  Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

## Step 5: Configure LoRA

LoRA adds trainable adapter layers instead of fine-tuning the entire model.

In [ ]:
# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
peft_config = LoraConfig(
    r=16,  # LoRA rank (higher = more parameters, better quality, slower)
    lora_alpha=32,  # LoRA scaling
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Which layers to adapt
)

# Add LoRA adapters
model = get_peft_model(model, peft_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"\nTrainable parameters: {trainable_params / 1e6:.2f}M ({100 * trainable_params / total_params:.2f}%)")
print(f"Total parameters: {total_params / 1e6:.2f}M")
print(f"\n✓ LoRA adapters added - only training {trainable_params / 1e6:.2f}M parameters!")

## Step 6: Configure Training

Set hyperparameters for fine-tuning.

In [ ]:
# Output directory
output_dir = project_root / "models/finetuned_llm"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_output = output_dir / f"llama_finetuned_{timestamp}"
model_output.mkdir(parents=True, exist_ok=True)

# Training arguments
training_args = TrainingArguments(
    output_dir=str(model_output),
    num_train_epochs=3,
    per_device_train_batch_size=4,  # Reduce if OOM
    gradient_accumulation_steps=4,  # Effective batch size = 4*4 = 16
    per_device_eval_batch_size=4,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.01,
    fp16=True,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",  # Disable wandb/tensorboard
)

print("Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Total steps: ~{len(train_dataset) * training_args.num_train_epochs // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")

## Step 7: Initialize Trainer

Use HuggingFace's SFTTrainer (Supervised Fine-Tuning Trainer) optimized for instruction tuning.

In [ ]:
# Initialize trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args,
)

print("✓ Trainer initialized")
print("\nReady to start training!")

## Step 8: Train the Model!

**This is where your custom LLM is born!**

**Expected time:**
- With GPU (T4/V100): 2-4 hours
- Monitor: Loss should decrease steadily

**🍿 Time for a movie while the model trains!**

In [ ]:
print(f"{'='*80}")
print("STARTING TRAINING")
print(f"{'='*80}\n")
print(f"Model will be saved to: {model_output}")
print(f"Estimated time: 2-4 hours\n")
print(f"{'='*80}\n")

# Train!
trainer.train()

print(f"\n{'='*80}")
print("✓ TRAINING COMPLETE!")
print(f"{'='*80}\n")

## Step 9: Save the Model

In [ ]:
# Save model and tokenizer
trainer.model.save_pretrained(model_output)
tokenizer.save_pretrained(model_output)

print(f"✓ Model saved to: {model_output}")

# Save training info
training_info = {
    'base_model': MODEL_NAME,
    'training_date': timestamp,
    'num_training_examples': len(train_dataset),
    'num_epochs': training_args.num_train_epochs,
    'final_loss': trainer.state.log_history[-1].get('loss', 'N/A'),
    'lora_rank': peft_config.r,
    'trainable_params': f"{trainable_params / 1e6:.2f}M"
}

with open(model_output / "training_info.json", 'w') as f:
    json.dump(training_info, f, indent=2)

print("✓ Training info saved")

## Step 10: Test Your Custom LLM!

In [ ]:
# Create inference pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.3,
    top_p=0.95,
    repetition_penalty=1.15
)

def ask_model(question, context=""):
    """Ask your fine-tuned model a question."""
    prompt = (
        f"[INST] <<SYS>>\n"
        f"You are a helpful research assistant.\n"
        f"<</SYS>>\n\n"
        f"Question: {question}\n"
    )
    
    if context:
        prompt += f"Context: {context}\n"
    
    prompt += "[/INST]\n\nAnswer:"
    
    result = pipe(prompt)[0]['generated_text']
    
    # Extract answer (after "Answer:")
    answer = result.split("Answer:")[-1].strip()
    
    return answer

# Test questions
test_questions = [
    "What is the main contribution of this research?",
    "What sensors were used in the drone?",
    "How does Visual SLAM work?"
]

print(f"{'='*80}")
print("TESTING YOUR CUSTOM LLM")
print(f"{'='*80}\n")

for q in test_questions:
    print(f"Q: {q}")
    answer = ask_model(q)
    print(f"A: {answer}\n")
    print("-" * 80 + "\n")

## Step 11: Compare with Base Model

Let's see the improvement!

In [ ]:
# Load base model for comparison
print("Loading base model for comparison...\n")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

base_pipe = pipeline(
    "text-generation",
    model=base_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.3
)

# Test same question
test_q = "What sensors were used in the drone experiment?"
test_context = "The Intel RealSense D435i depth camera and T265 tracking camera were used."

print(f"{'='*80}")
print("COMPARISON: Base vs Fine-tuned")
print(f"{'='*80}\n")
print(f"Question: {test_q}\n")
print(f"Context: {test_context}\n")
print("-" * 80 + "\n")

# Base model
base_prompt = f"Question: {test_q}\nContext: {test_context}\nAnswer:"
base_answer = base_pipe(base_prompt)[0]['generated_text'].split("Answer:")[-1].strip()

print("BASE MODEL:")
print(base_answer[:200])
print("\n" + "-" * 80 + "\n")

# Fine-tuned model
finetuned_answer = ask_model(test_q, test_context)

print("YOUR FINE-TUNED MODEL:")
print(finetuned_answer[:200])
print("\n" + "=" * 80)

print("\nNotice how your fine-tuned model gives more focused, research-specific answers!")

## 🎉 Congratulations!

You now have:
- ✅ Custom LLM fine-tuned on YOUR research papers
- ✅ Runs locally (no API costs!)
- ✅ Better domain-specific answers
- ✅ Only 16MB of LoRA adapters (vs 14GB full model)

## Next Steps:

### 1. Use in Your RAG Pipeline:

```python
# Create custom LLM handler that loads your model
from transformers import pipeline
from peft import PeftModel

# Load your fine-tuned model
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, ...)
model = PeftModel.from_pretrained(base, str(model_output))

# Use in RAG
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
```

### 2. Evaluate Performance:
- Compare with Gemini API
- Measure answer quality (ROUGE, semantic similarity)
- Test on held-out questions

### 3. Deploy:
- Run inference server locally
- Integrate with Streamlit UI
- No more API costs!

---

**You've built a complete, custom RAG system with:**
- Custom embeddings (better retrieval)
- Custom LLM (better generation)
- Hybrid search (best accuracy)

**This is production-grade, research-quality work! 🚀**